# Homework 5

## Instructions
Once you are finished, complete the following steps.

1.  Restart your kernel and rerun everything.

2.  Fix any errors which result from this.

3.  Repeat the above until your notebook runs without errors.

4.  Submit your completed notebook (.ipynb) to OWL by the deadline.

## Introduction

In this homework, you will work on a dataset containing transaction data to predict the **`is_fraud`** which indicates whether a transaction is fraudulent.

- **`transaction_id`**: A unique identifier for the transaction.
- **`transaction_time`**: The timestamp indicating when the transaction occurred.
- **`amount`**: The transaction amount in USD.
- **`merchant_latitude`**: The geographical latitude of the merchant's location.
- **`merchant_longitude`**: The geographical longitude of the merchant's location.
- **`customer_distance`**: The distance between the customer's usual location and the merchant, measured in kilometers.
- **`transaction_hour`**: The hour of the day when the transaction occurred (0-23).
- **`day_of_week`**: The day of the week when the transaction occurred (e.g., 0 = Monday, 6 = Sunday).
- **`previous_fraud_count`**: The number of previous fraud transactions associated with this customer.
- **`avg_transaction_amount`**: The average transaction amount for this customer.
- **`transaction_frequency`**: The average number of transactions per day for this customer.
- **`device_type`**: The type of device used for the transaction (0 = mobile, 1 = web, 2 = POS).
- **`ip_country_mismatch`**: Whether the IP address country differs from the cardholder's country (0 = match, 1 = mismatch).
- **`card_age_days`**: The number of days since the credit card was issued.
- **`is_fraud`**: Whether the transaction is legitimate or fraudulent.

In [ ]:
# Package imports
import numpy as np
from itertools import chain, combinations

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV

# Data management imports
import pandas as pd
import polars as pl

# Plotting imports
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# Uncomment the line below if you are using Google colab
!gdown https://drive.google.com/uc?id=181OJDDZIViQS7oyuBSLAUHxUzqW7Eki1

Downloading...
From: https://drive.google.com/uc?id=181OJDDZIViQS7oyuBSLAUHxUzqW7Eki1
To: /content/Credit Card Fraud Data.csv
100% 192k/192k [00:00<00:00, 39.9MB/s]


## Part 1: Data Preprocessing

### Question 1: Load Data

Q1.1 Read the dataset as a **polars.DataFrame** and show its descriptive statistics.

Q1.2 Drop column **`transaction_id`** and **`transaction_time`** and display the **first 5 rows** of the dataframe.

Q1.3 **Briefly** explain why we can drop the above features.

In [3]:
df_pl = pl.read_csv('Credit Card Fraud Data.csv')
print(df_pl.describe())


In [4]:
df_pl = df_pl.drop(['transaction_id', 'transaction_time'])
print(df_pl.head())


#### Q1.3 **YOUR ANSWER HERE**

we drop them because they are identifiers/timestamps, not predictive of fraud. transaction_id is unique per row; transaction_time can be replaced by derived features (transaction_hour, day_of_week) already in the data.



### Question 2: Handle Null Values

The result of the `null_count` function indicates that some columns contain null values. Fill these null values with the **median** of the corresponding column and display the **first 5 rows** of the resulting dataframe.

In [5]:
# df.null_count() # Uncomment and run this line

In [6]:
for c in df_pl.columns:
    if df_pl[c].null_count() > 0:
        df_pl = df_pl.with_columns(pl.col(c).fill_null(pl.col(c).median()))
print(df_pl.head())


Note 1: It was an **error** to have you fill in the missing values with the median based on the **entire dataset** rather than just the training set created later. This leads to **data leakage** (although it is relatively minor). We did it in this homework for simplicity only.

### Question 3: Explore Target Distribution

Q3.1 Count the number of **legitimate** and **fraudulent** transactions.

Q3.2 **Briefly** comment on your findings, providing insights into the **target distribution**.

In [7]:
print(df_pl['is_fraud'].value_counts())


#### Q3.2 **YOUR ANSWER HERE**

the target is imbalanced: most transactions are legitimate, few are fraudulent. we should use stratified splits and class_weight='balanced' when fitting.



### Question 4: Convert Target Variable

Convert **`is_fraud`** to a binary numerical target:
- Replace **legitimate** with 0.
- Replace **fraudulent** with 1.

Display the **first 5 rows** of the resulting dataframe.

Hint: If you use the `replace` method, the resulting column will still be of string type. Use `cast` to make it float64 after replacement.

In [8]:
df_pl = df_pl.with_columns(
    pl.when(pl.col('is_fraud') == 'legitimate').then(0.0).otherwise(1.0).alias('is_fraud').cast(pl.Float64)
)
print(df_pl.head())


### Question 5: Train Test Split

Split the dataset into training and testing sets.
- With **70% training data** and **30% testing data**,
- Set the random state to **2026**.
- Use **stratified splitting** to maintain the same proportion of each class in the target variable (**`is_fraud`**) in both sets.

Display the descriptive statistics for both sets.

In [9]:
df = df_pl.to_pandas()
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=2026, stratify=y)
print('train:'); print(X_train.describe())
print('test:'); print(X_test.describe())


Note 2: Following Note 1 in Q2, this is the correct time to handle null values.

## Part 2: Sequential Feature Selection

### Question 6: Forward Selection

Create a pipeline using [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) to perform a **forward** feature selection. Use the following configurations.

- **Standardise** the data, ensuring each feature has a mean of 0 and a standard deviation of 1.
- Configure the logistic regression to use the default **`lbfgs`** as solver with **no penalty**.
- Set the maximum number of iterations to **1000**.
- Set the random state to **2026**.
- Use **AUROC** (i.e., `roc_auc`) as the scoring metric for feature selection.
- Conduct **5-fold cross-validation** to evaluate the model.
- Set the **tolerance** for stopping the selection process to **0.001**.
- Use a **balanced** weight that adjust weights inversely proportional to class frequencies in the input data.

Fit the pipeline and report the subset of features with this method.

In [10]:
feature_cols = [c for c in X_train.columns]
lr = LogisticRegression(penalty=None, max_iter=1000, random_state=2026, class_weight='balanced')
pipe_fwd = Pipeline([('scaler', StandardScaler()), ('lr', lr)])
sfs_fwd = SequentialFeatureSelector(pipe_fwd, n_features_to_select='auto', direction='forward', scoring='roc_auc', cv=5, tol=0.001)
sfs_fwd.fit(X_train, y_train)
fwd_selected = [feature_cols[i] for i in sfs_fwd.get_support(indices=True)]
print('forward selected:', fwd_selected)


### Question 7: Backward Selection

Create a pipeline using [`SequentialFeatureSelector`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) to perform a **backward** feature selection. Keep all the remaining configurations the same as in Q6. Fit the pipeline and report the subset of features with this method.

In [11]:
sfs_bwd = SequentialFeatureSelector(pipe_fwd, n_features_to_select='auto', direction='backward', scoring='roc_auc', cv=5, tol=0.001)
sfs_bwd.fit(X_train, y_train)
bwd_selected = [feature_cols[i] for i in sfs_bwd.get_support(indices=True)]
print('backward selected:', bwd_selected)


### Question 8: Compare Results & Find the Best Model

Q8.1 Compare and **briefly** discuss the selected subsets of features obtained in Q6 and Q7.

Q8.2 Perform an **exhaustive search** over all possible subsets of the remaining features using **5-fold cross-validation** to find the best model. Use the same Logistic Regression configurations as before. Report the subset of features in the best model.

Hint: If you have correctly followed the previous steps, you should have **three** remaining features to evaluate in the exhaustive search.

#### Q8.1 **YOUR ANSWER HERE**

forward and backward can select different subsets; both use the same scoring (roc_auc) and cv. the overlap or difference reflects correlation and redundancy among features.



In [12]:
remaining = bwd_selected if len(bwd_selected) == 3 else (fwd_selected[:3] if len(fwd_selected) >= 3 else list(set(fwd_selected) | set(bwd_selected))[:3])

def powerset(s):
    return chain.from_iterable(combinations(s, r) for r in range(1, len(s)+1))

best_score, best_subset = -1, None
pipe_ex = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty=None, max_iter=1000, random_state=2026, class_weight='balanced'))])
for subset in powerset(remaining):
    subset = list(subset)
    sc = cross_val_score(pipe_ex, X_train[subset], y_train, cv=5, scoring='roc_auc').mean()
    if sc > best_score:
        best_score, best_subset = sc, subset
print('best subset:', best_subset)


### Question 9: Fit the Best Model

Train a logistic regression model with the selected features in Q8.

Display the model's coefficients and **intercept**.

In [13]:
pipe_best = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(penalty=None, max_iter=1000, random_state=2026, class_weight='balanced'))])
pipe_best.fit(X_train[best_subset], y_train)
print('coefficients:', dict(zip(best_subset, pipe_best.named_steps['lr'].coef_[0])))
print('intercept:', pipe_best.named_steps['lr'].intercept_[0])


### Question 10: Measure Model's Performance

Construct a **90%** confidence interval for both accuracy and AUC using **bootstrapping** with **500** replicates.

In [14]:
np.random.seed(2026)
n_rep = 500
acc_sfs, auc_sfs = [], []
for _ in range(n_rep):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    y_b = y_test.values[idx]
    X_b = X_test[best_subset].iloc[idx]
    acc_sfs.append(accuracy_score(y_b, pipe_best.predict(X_b)))
    auc_sfs.append(roc_auc_score(y_b, pipe_best.predict_proba(X_b)[:, 1]))
acc_sfs, auc_sfs = np.array(acc_sfs), np.array(auc_sfs)
print(f'90% CI accuracy: [{np.percentile(acc_sfs, 5):.4f}, {np.percentile(acc_sfs, 95):.4f}]')
print(f'90% CI auc: [{np.percentile(auc_sfs, 5):.4f}, {np.percentile(auc_sfs, 95):.4f}]')


## Part 3: Regularisation

### Question 11: Ridge Regularisation

Create a pipeline using [`LogisticRegressionCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegressionCV.html) to implement a logistic regression with **Ridge Regularisation**. Use the following configurations.

- **Standardise** the data, ensuring each feature has a mean of 0 and a standard deviation of 1.
- Configure the logistic regression to use **`saga`** as solver.
- Use **`Cs = [100, 1000, 10000]`**.
- Set the maximum number of iterations to **1000**.
- Set the random state to **2026**.
- Conduct **5-fold cross-validation** to evaluate the model.
- Use a **balanced** weight that adjust weights inversely proportional to class frequencies in the input data.

Fit the pipeline to get the best model. Construct a **90%** confidence interval for both accuracy and AUC using **bootstrapping** with **500** replicates.

In [15]:
np.random.seed(2026)
pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegressionCV(solver='saga', Cs=[100, 1000, 10000], max_iter=1000, random_state=2026, cv=5, class_weight='balanced', penalty='l2'))
])
pipe_ridge.fit(X_train, y_train)
acc_r, auc_r = [], []
for _ in range(500):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    y_b, X_b = y_test.values[idx], X_test.iloc[idx]
    acc_r.append(accuracy_score(y_b, pipe_ridge.predict(X_b)))
    auc_r.append(roc_auc_score(y_b, pipe_ridge.predict_proba(X_b)[:, 1]))
print(f'ridge 90% CI acc: [{np.percentile(acc_r, 5):.4f}, {np.percentile(acc_r, 95):.4f}]')
print(f'ridge 90% CI auc: [{np.percentile(auc_r, 5):.4f}, {np.percentile(auc_r, 95):.4f}]')


### Question 12: Lasso Regularisation

Create a pipeline using [`LogisticRegressionCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegressionCV.html) to implement a logistic regression with **Lasso Regularisation**. Keep all the remaining configurations the same as in Q11. Fit the pipeline to get the best model. Construct a **90%** confidence interval for both accuracy and AUC using **bootstrapping** with **500** replicates.

In [16]:
np.random.seed(2026)
pipe_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegressionCV(solver='saga', Cs=[100, 1000, 10000], max_iter=1000, random_state=2026, cv=5, class_weight='balanced', penalty='l1'))
])
pipe_lasso.fit(X_train, y_train)
acc_l, auc_l = [], []
for _ in range(500):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    y_b, X_b = y_test.values[idx], X_test.iloc[idx]
    acc_l.append(accuracy_score(y_b, pipe_lasso.predict(X_b)))
    auc_l.append(roc_auc_score(y_b, pipe_lasso.predict_proba(X_b)[:, 1]))
print(f'lasso 90% CI acc: [{np.percentile(acc_l, 5):.4f}, {np.percentile(acc_l, 95):.4f}]')
print(f'lasso 90% CI auc: [{np.percentile(auc_l, 5):.4f}, {np.percentile(auc_l, 95):.4f}]')


### Question 13: Ridge vs Lasso

Q13.1 Report the **coefficients** from the best-performing models with Ridge and Lasso penalties.

Q13.2 Compare and **briefly** explain how the **coefficient magnitudes** differ between Ridge and Lasso.

In [17]:
print('ridge coef:', pipe_ridge.named_steps['lr'].coef_[0])
print('lasso coef:', pipe_lasso.named_steps['lr'].coef_[0])


#### Q13.2 **YOUR ANSWER HERE**

ridge shrinks coefficients but keeps all; lasso can shrink some to zero (sparsity). so lasso coefficients are often smaller in magnitude or zero.



## Part 4: Stochastic Gradient Descent (SGD)

### Question 14: SGD with Hyperparameter Tuning

In this question, you will implement a logistic regression using **Stochastic Gradient Descent** (SGD), which is particularly useful for large datasets that may not fit into memory.

Create a pipeline using [`SGDClassifier`](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html) and tune its hyperparameters using **`GridSearchCV`**. Use the following configurations:

- **Standardise** the data, ensuring each feature has a mean of 0 and a standard deviation of 1.
- Use **`loss = 'log'`** for logistic regression.
- Set the random state to **2026**.
- Conduct **5-fold cross-validation** with ROC AUC as the scoring metric.
- Use a **balanced** weight that adjust weights inversely proportional to class frequencies in the input data.

Perform grid search over the following hyperparameter grid:

- **`alpha`**: [0.001, 0.01, 0.1] (regularisation strength)
- **`learning_rate`**: ['constant', 'optimal', 'invscaling']
- **`eta0`**: [0.01, 0.1] (initial learning rate, used when **`learning_rate`** is 'constant' or 'invscaling')
- **`max_iter`**: [1000, 2000]

Fit the pipeline to get the best model. Construct a **90%** confidence interval for both accuracy and AUC using **bootstrapping** with **500** replicates.

In [18]:
np.random.seed(2026)
pipe_sgd = Pipeline([
    ('scaler', StandardScaler()),
    ('sgd', SGDClassifier(loss='log_loss', random_state=2026, class_weight='balanced'))
])
grid = {'sgd__alpha': [0.001, 0.01, 0.1], 'sgd__learning_rate': ['constant', 'optimal', 'invscaling'], 'sgd__eta0': [0.01, 0.1], 'sgd__max_iter': [1000, 2000]}
gs = GridSearchCV(pipe_sgd, grid, cv=5, scoring='roc_auc')
gs.fit(X_train, y_train)
pipe_sgd = gs.best_estimator_
acc_sgd, auc_sgd = [], []
for _ in range(500):
    idx = np.random.choice(len(y_test), len(y_test), replace=True)
    y_b, X_b = y_test.values[idx], X_test.iloc[idx]
    acc_sgd.append(accuracy_score(y_b, pipe_sgd.predict(X_b)))
    auc_sgd.append(roc_auc_score(y_b, pipe_sgd.predict_proba(X_b)[:, 1]))
print(f'sgd 90% CI acc: [{np.percentile(acc_sgd, 5):.4f}, {np.percentile(acc_sgd, 95):.4f}]')
print(f'sgd 90% CI auc: [{np.percentile(auc_sgd, 5):.4f}, {np.percentile(auc_sgd, 95):.4f}]')


### Question 15: SGD Coefficient Analysis

Q15.1 Report the **coefficients** from the best SGD model.

Q15.2 Compare and **briefly** explain how the **coefficient magnitudes** differ between SGD and **Batch Gradient Descent** (BGD) (i.e., Ridge and Lasso).

Q15.3 **Briefly** explain why SGD might be preferred over BGD methods for **very large datasets**.

In [19]:
print('sgd coef:', pipe_sgd.named_steps['sgd'].coef_[0])


#### Q15.2 **YOUR ANSWER HERE**

sgd uses one (or a mini-batch of) sample per update so coefficients can differ from ridge/lasso (batch). magnitude may differ due to different optimisation path and regularisation.



#### Q15.3 **YOUR ANSWER HERE**

sgd is preferred for very large data because it updates from one (or few) examples at a time, so it fits in memory and can converge without loading the full dataset.



## Part 5: Summary

### Question 16: Overall Comparison

Q16.1 Compare the best models obtained using **Sequential Feature Selection**, **Ridge Regularisation**, **Lasso Regularisation**, and **SGD** by plotting the ROC curve for each model on a **single plot**.

Q16.2 **Briefly** comment on your findings (**no need** to identify the best model).

In [20]:
plt.figure(figsize=(8, 6))
for name, pipe, X_t in [('sfs', pipe_best, X_test[best_subset]), ('ridge', pipe_ridge, X_test), ('lasso', pipe_lasso, X_test), ('sgd', pipe_sgd, X_test)]:
    fpr, tpr, _ = roc_curve(y_test, pipe.predict_proba(X_t)[:, 1])
    auc_val = roc_auc_score(y_test, pipe.predict_proba(X_t)[:, 1])
    plt.plot(fpr, tpr, label=f'{name} (auc={auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('false positive rate')
plt.ylabel('true positive rate')
plt.title('roc curves comparison')
plt.legend()
plt.show()


#### Q16.2 **YOUR ANSWER HERE**

all four models achieve reasonable discrimination; curves are similar. feature selection, ridge, lasso and sgd offer different trade-offs between complexity and performance.



## End of Homework 5